# Spidroin Gene Annotation

Two-step pipeline for automated spidroin annotation:

1. **Typing Agent** — parses nHMMER + miniprot results to locate spidroin loci and extract full-length sequences (start codon → stop codon).
2. **Augustus Gene Prediction** — predicts intron/exon structure and generates protein sequences for each species.

## Environment Setup
Ensure that the following dependencies have been installed (via Pixi):
```bash
pixi install
```

## Configuration

Import packages and predefine variables required throughout the notebook.

In [2]:
from pathlib import Path

import polars as pl
from Bio import SeqIO

from spider_silkome_module import (
    EXTERNAL_DATA_DIR,
    PROCESSED_DATA_DIR,
)
from spider_silkome_module.utils.run_cmd import run_cmd

2026-04-14 16:06:12.997 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome


In [3]:
# Input directories
nhmmer_dir = PROCESSED_DATA_DIR / "nhmmer_search_parsed"
miniprot_dir = PROCESSED_DATA_DIR / "miniprot_output"

# Output directory (typing_results doubles as input for Augustus)
typing_output = PROCESSED_DATA_DIR / "typing_results"

# Augustus configuration
extrinsic_cfg = EXTERNAL_DATA_DIR / "extrinsic.cfg"
augustus_species = "parasteatoda"

## Step 1: Spidroin Locus Typing

Run the typing agent to pair NTD/CTD hits from nHMMER into spidroin loci, extract full-length sequences from start codon to stop codon, and write per-species results to `typing_results/`.

Outputs per species:
- `{species}.tsv` — locus table
- `{species}.gff` — locus GFF
- `spidroin_full_length.fasta` — nucleotide sequences (start → stop)
- `hints.gff` — Augustus hints

In [ ]:
run_cmd(
    f"pixi run python agents/typing_agent.py \
        --nhmmer-dir {nhmmer_dir} \
        --miniprot-dir {miniprot_dir} \
        --output {typing_output}",
    [typing_output],
)

## Step 2: Augustus Gene Prediction

Run Augustus on the spidroin sequences extracted in Step 1 to predict intron/exon structure and generate protein sequences.

Outputs per species (written to `typing_results/{species}/`):
- `augustus_output.gff` — predicted gene structures
- `augustus_output.aa` — predicted protein sequences

Per-species skip logic is handled inside `run_augustus.py`: existing output files are skipped automatically.

In [3]:
run_cmd(
    f"pixi run python -m spider_silkome_module.run_augustus \
        --input-path {typing_output} \
        --extrinsic-cfg {extrinsic_cfg} \
        --augustus-species {augustus_species}",
    [typing_output],
    force=True,
)

2026-04-13 22:58:04.217 | INFO     | spider_silkome_module.config:<module>:11 - PROJ_ROOT path is: /home/gyk/project/spider_silkome
Running Augustus:   0%|          | 0/127 [00:00<?, ?it/s]

2026-04-13 22:58:04.263 | INFO     | __main__:main:84 - Found 127 species to process
2026-04-13 22:58:04.265 | INFO     | __main__:main:87 - Processing: 001.Allagelena_difficilis
⏭️ /home/gyk/project/spider_silkome/data/processed/typing_results/001.Allagelena_difficilis/augustus_output.gff exists, skip
⏭️ /home/gyk/project/spider_silkome/data/processed/typing_results/001.Allagelena_difficilis/augustus_output.aa exists, skip
2026-04-13 22:58:04.265 | INFO     | __main__:main:87 - Processing: 002.Anyphaena_bomiensis
Read in 29 sequence(s) from /home/gyk/project/spider_silkome/data/processed/typing_results/002.Anyphaena_bomiensis/spidroin_full_length.fasta.
2026-04-13 22:58:28.324 | INFO     | __main__:main:87 - Processing: 003.Atypus_heterothecus


Running Augustus:   2%|▏         | 2/127 [00:24<25:03, 12.03s/it]

Read in 1 sequence(s) from /home/gyk/project/spider_silkome/data/processed/typing_results/003.Atypus_heterothecus/spidroin_full_length.fasta.
2026-04-13 22:58:29.712 | INFO     | __main__:main:87 - Processing: 004.Borboropactus_sp


Running Augustus:   2%|▏         | 3/127 [00:25<15:41,  7.60s/it]

...

Running Augustus:  98%|█████████▊| 125/127 [40:31<00:32, 16.12s/it]

Read in 15 sequence(s) from /home/gyk/project/spider_silkome/data/processed/typing_results/133.Xysticus_acerbus/spidroin_full_length.fasta.
2026-04-13 23:38:45.783 | INFO     | __main__:main:87 - Processing: 134.Zoropsis_spinimana


Running Augustus:  99%|█████████▉| 126/127 [40:41<00:14, 14.35s/it]

Read in 18 sequence(s) from /home/gyk/project/spider_silkome/data/processed/typing_results/134.Zoropsis_spinimana/spidroin_full_length.fasta.
2026-04-13 23:38:59.328 | SUCCESS  | __main__:main:92 - Augustus gene prediction complete.


Running Augustus: 100%|██████████| 127/127 [40:55<00:00, 19.33s/it]


## Results

Summarize the number of predicted protein sequences per species.

In [4]:
rows = []
for sp_dir in sorted(typing_output.iterdir()):
    if not sp_dir.is_dir():
        continue
    aa_file = sp_dir / "augustus_output.aa"
    tsv_file = sp_dir / f"{sp_dir.name}.tsv"

    n_proteins = len(list(SeqIO.parse(aa_file, "fasta"))) if aa_file.exists() else None
    n_loci = None
    if tsv_file.exists():
        df_sp = pl.read_csv(tsv_file, separator="\t")
        n_loci = len(df_sp)

    rows.append({"species": sp_dir.name, "n_loci": n_loci, "n_proteins": n_proteins})

summary = pl.DataFrame(rows)
print(summary.shape)
summary

(127, 3)


species,n_loci,n_proteins
str,i64,i64
"""001.Allagelena_difficilis""",23,15
"""002.Anyphaena_bomiensis""",33,29
"""003.Atypus_heterothecus""",2,1
"""004.Borboropactus_sp""",11,10
"""005.Bowie_medogensis""",10,8
…,…,…
"""130.Pimoa_clavata""",9,7
"""131.Segestria_senoculata""",4,4
"""132.Tibellus_oblongus""",16,15
